# Scientific figures: editable source notebook

Each executable cell below produces exactly one complete main or supplementary figure. Every cell imports its libraries, reads the required project results, performs the filtering and aggregation, specifies the visual properties, and exports a PNG. No cell imports or calls a separate plotting module.

Run the notebook from the repository root or from the `plot` directory. All 23 figures are written to `output/figures`, using Figure 1–6 and Supplementary Figure S1–S17 numbering.


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# FIGURE 1 — every operation for this figure is in this cell.
catalog = pd.read_csv(SD / "Figure1a_heatwave_catalog.csv", parse_dates=["start_date", "end_date"])
annual = pd.read_csv(SD / "Figure1b_HKO_annual_heat_metrics.csv")
future = pd.read_csv(SD / "Figure1c_future_climate_background.csv")
tmax = pd.read_csv(SD / "Figure1d_Tmax_holdout.csv")
tmin = pd.read_csv(SD / "Figure1e_Tmin_holdout.csv")
grid = pd.read_csv(SD / "Figure1f_spatial_hazard_grid.csv")

fig = plt.figure(figsize=(10.58, 9.97), constrained_layout=False)
outer = fig.add_gridspec(3, 2, left=.060, right=.955, bottom=.055, top=.970,
                         wspace=.285, hspace=.355)

# (a) Finalized bubble-calendar grammar; turquoise rings identify all eight selected events.
ax = fig.add_subplot(outer[0, 0])
month_names = {5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct"}
sizes = 12 + 4.1 * catalog.duration_days.clip(lower=1)
sc = ax.scatter(catalog.start_year, catalog.start_month, s=sizes,
                c=catalog.mean_intensity_c, cmap="magma", alpha=.82,
                edgecolor="white", linewidth=.35)
selected = catalog[catalog.event_id.isin(EVENT_IDS)]
ax.scatter(selected.start_year, selected.start_month,
           s=28 + 4.6 * selected.duration_days, facecolors="none", edgecolors="#00A6A6",
           linewidths=1.8, zorder=5)
ax.set_yticks(sorted(month_names), [month_names[m] for m in sorted(month_names)])
ax.set_ylim(4.65, 11.20)  # reserve a full month-equivalent band above October for the legend
ax.set_xlabel("Event start year"); ax.set_ylabel("Start month")
ax.grid(axis="x", color="#D9DDE0", lw=.55, alpha=.55)
handles = [plt.scatter([],[],s=12+4.1*d,color="#7E3F7E",label=f"{d} d") for d in (3,10,20)]
handles += [plt.scatter([],[],s=70,facecolors="none",edgecolors="#00A6A6",lw=1.8,
                        label="Eight building-stress events")]
ax.legend(handles=handles, frameon=False, loc="upper left", ncol=2,
          columnspacing=.8, handletextpad=.45)
cb=fig.colorbar(sc,ax=ax,fraction=.048,pad=.018); cb.set_label("Mean Tmax excess (°C)")
panel_title(ax,"a","Observed HKO heatwaves, 1980–2025")

# (b) Frequency, event span and cumulative intensity on aligned axes.
block = outer[0,1].subgridspec(3,1,hspace=.08)
trend_specs=[("li_event_count","Events\n(count yr⁻¹)","#2F75B5"),
             ("li_event_span_days","Event span\n(days yr⁻¹)","#E45756"),
             ("li_cumulative_intensity_c_days","Cumulative\nintensity (°C d yr⁻¹)","#7A5195")]
for i,(field,ylabel,colour) in enumerate(trend_specs):
    bx=fig.add_subplot(block[i,0]); x=annual.year.to_numpy(float); y=annual[field].fillna(0).to_numpy(float)
    slope,intercept=np.polyfit(x,y,1); residual=y-(slope*x+intercept)
    se=np.sqrt(np.sum(residual**2)/max(len(x)-2,1)/np.sum((x-x.mean())**2))
    bx.plot(x,y,color=colour,alpha=.48,lw=.9,marker="o",ms=2.4)
    bx.plot(x,slope*x+intercept,color=colour,lw=2.0)
    bx.text(.02,.83,f"slope {slope:+.3f} ± {1.96*se:.3f}",transform=bx.transAxes,
            color=colour,ha="left",va="top")
    bx.set_ylabel(ylabel); bx.grid(axis="y",color="#D9DDE0",lw=.5,alpha=.55)
    if i<2: bx.tick_params(labelbottom=False)
    else: bx.set_xlabel("Year")
    if i==0: panel_title(bx,"b","Annual frequency, span and cumulative intensity")

# (c) Climate-only five-model ensemble; no incompatible future-building output is used.
ax=fig.add_subplot(outer[1,0])
scenario_colour={"ssp245":"#3B8BC2","ssp585":"#E66B6B"}
period_marker={"mid_century_2041_2060":"o","late_century_2081_2100":"^"}
for scenario in scenario_colour:
    rows=[]
    for period in period_marker:
        q=future[(future.experiment_id.eq(scenario)) & future.future_period.eq(period)]
        dx=q[q.metric.eq("delta_p95_tmax_c")].iloc[0]; hy=q[q.metric.eq("very_hot_days_per_summer")].iloc[0]
        rows.append((dx.ensemble_p50,hy.ensemble_p50))
        ax.plot([dx.ensemble_p10,dx.ensemble_p90],[hy.ensemble_p50]*2,color=scenario_colour[scenario],alpha=.35,lw=1)
        ax.plot([dx.ensemble_p50]*2,[hy.ensemble_p10,hy.ensemble_p90],color=scenario_colour[scenario],alpha=.35,lw=1)
        ax.scatter(dx.ensemble_p50,hy.ensemble_p50,s=62,marker=period_marker[period],
                   color=scenario_colour[scenario],edgecolor="white",lw=.7,zorder=3)
    ax.plot(*zip(*rows),color=scenario_colour[scenario],lw=1.3,alpha=.65)
legend=[Line2D([],[],color=c,marker="o",label=s.upper()) for s,c in scenario_colour.items()]
period_labels={"mid_century_2041_2060":"2041–2060","late_century_2081_2100":"2081–2100"}
legend += [Line2D([],[],color="#555",marker=m,ls="",label=period_labels[p]) for p,m in period_marker.items()]
ax.legend(handles=legend,frameon=False,loc="lower right",ncol=2)
ax.set_xlabel("Δ summer P95 Tmax vs 1992–2014 (°C)"); ax.set_ylabel("Very hot days per summer")
ax.grid(color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"c","Future heat-tail amplification across five models")

def paired_violin(container,data,letter,title):
    sub=container.subgridspec(3,1,hspace=.10)
    specs=[("raw_quantile_mae_c","corrected_quantile_mae_c","Quantile MAE\n(°C)","raw"),
           ("raw_p95_error_c","corrected_p95_error_c","|P95 error|\n(°C)","abs"),
           ("raw_threshold_fraction","corrected_threshold_fraction","Threshold error\n(percentage points)","threshold")]
    for j,(raw,corrected,ylabel,mode) in enumerate(specs):
        vx=fig.add_subplot(sub[j,0]); r=data[raw].to_numpy(float); c=data[corrected].to_numpy(float)
        if mode=="abs": r,c=np.abs(r),np.abs(c)
        if mode=="threshold":
            obs=data.observed_threshold_fraction.to_numpy(float); r=np.abs(r-obs)*100; c=np.abs(c-obs)*100
        vp=vx.violinplot([r,c],positions=[0,1],widths=.80,showmedians=True,showextrema=False)
        for body,col in zip(vp["bodies"],["#BBC5CF","#86C3BA"]):
            body.set_facecolor(col); body.set_edgecolor("none"); body.set_alpha(.72)
        vp["cmedians"].set_color("#1F3A4D"); vp["cmedians"].set_linewidth(1.2)
        for rr,cc in zip(r,c): vx.plot([0,1],[rr,cc],color="#A9B6BF",lw=.35,alpha=.20,zorder=0)
        vx.scatter([0,1],[np.median(r),np.median(c)],s=24,facecolor="white",edgecolor="#1F3A4D",zorder=4)
        improve=np.mean(c<r)*100; change=(np.median(c)-np.median(r))/max(np.median(r),1e-9)*100
        lo,hi=vx.get_ylim(); vx.set_ylim(lo, hi + (hi-lo)*.28)
        vx.text(.98,.94,f"median {change:+.0f}%  |  {improve:.0f}% improved",transform=vx.transAxes,ha="right",va="top")
        vx.set_ylabel(ylabel); vx.grid(axis="y",color="#D9DDE0",lw=.45,alpha=.50)
        vx.set_xticks([0,1],["Raw WRF","Bias corrected"] if j==2 else ["",""])
        if j==0: panel_title(vx,letter,title)
paired_violin(outer[1,1],tmax,"d","Independent Tmax validation")
paired_violin(outer[2,0],tmin,"e","Independent Tmin validation")

# (f) Each validated 1-km land cell is a square, matching the finalized map.
ax=fig.add_subplot(outer[2,1])
boundary_path=BOUNDARIES/"housing_market_areas.geoparquet"
if boundary_path.exists():
    try:
        boundary=gpd.read_parquet(boundary_path).to_crs(4326)
        boundary.plot(ax=ax,color="#D8DEDC",edgecolor="#87908F",linewidth=.25,zorder=0)
    except (OSError, ValueError):
        pass
values=grid.bias_corrected_tmax_p95_c.to_numpy(float)
ms=ax.scatter(grid.wrf_grid_longitude,grid.wrf_grid_latitude,c=values,marker="s",s=18,
              cmap="inferno",linewidths=0,vmin=np.nanpercentile(values,1),vmax=np.nanpercentile(values,99),zorder=2)
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)"); ax.set_aspect("equal",adjustable="box")
ax.set_xlim(grid.wrf_grid_longitude.min()-.025,grid.wrf_grid_longitude.max()+.025)
ax.set_ylim(grid.wrf_grid_latitude.min()-.018,grid.wrf_grid_latitude.max()+.018)
cb=fig.colorbar(ms,ax=ax,fraction=.040,pad=.018); cb.set_label("Warm-season P95 Tmax (°C)")
ax.set_anchor("W")  # align the aspect-constrained map's left y-axis spine with panels b and d
cbar_box=cb.ax.get_position(); cb.ax.set_position([cbar_box.x0-.008,cbar_box.y0,cbar_box.width,cbar_box.height])
panel_title(ax,"f","Observation-constrained 1-km P95 Tmax, 2011–2020")
finish(fig,"Figure_1_heatwave_climate_context")


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# FIGURE 2 — every operation for this figure is in this cell.
monthly=pd.read_csv(SD/"Figure2_monthly_shape.csv")
registry=pd.read_csv(SD/"Figure2_registry.csv")
population=pd.read_csv(SD/"Figure2_estate_population.csv")
heat=pd.read_csv(SD/"Figure2_heat_sensitivity.csv")
bca=pd.read_csv(SD/"Figure2_BCA_EUI.csv")
energy=pd.read_csv(SD/"Figure2_annual_energy.csv")

fig,axs=plt.subplots(3,2,figsize=(10.58,10.87),constrained_layout=False)
fig.subplots_adjust(left=.185,right=.975,bottom=.060,top=.970,wspace=.54,hspace=.44)

ax=axs[0,0]
x=monthly.month.to_numpy(); observed=monthly.official_share.to_numpy()*100; modeled=monthly.modeled_share.to_numpy()*100
ax.plot(x,observed,"o-",color="#214F77",lw=1.8,ms=4.5,label="Official 2023")
ax.plot(x,modeled,"D-",color="#D97706",lw=1.7,ms=4.0,label="Model holdout")
ax.set_xticks(range(1,13)); ax.set_xlabel("Month"); ax.set_ylabel("Share of annual domestic\nelectricity (%)")
ax.text(.04,.93,"r = 0.926\nCVRMSE = 12.8%",transform=ax.transAxes,ha="left",va="top")
ax.set_ylim(2.5,17.5); ax.legend(frameon=False,loc="upper right",bbox_to_anchor=(.985,.985)); ax.grid(color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"a","Official Hong Kong monthly demand shape")

# Separate observed-load benchmarks. The direct EnergyPlus concordance remains panel d.
ax=axs[0,1]; points=[]
for label,dataset,key in [("HKUST\nbuildings","HKUST complete archive","held-out"),
                          ("PolyU\nbuildings","PolyU full campus","building"),
                          ("PolyU\nfinal 28 d","PolyU full campus","time")]:
    row=registry[registry.dataset.eq(dataset)].iloc[0]
    nums=[float(v) for v in re.findall(r"[-+]?\d*\.?\d+",str(row.value))]
    if dataset.startswith("HKUST"): nmbe,cvrmse=nums[0],nums[1]
    elif key=="building": nmbe,cvrmse=nums[0],nums[1]
    else: nmbe,cvrmse=nums[2],nums[3]
    points.append((label,abs(nmbe),cvrmse))
label_offsets=[(-8,10),(8,-2),(8,10)]
for ((label,nmbe,cvrmse),colour,offset) in zip(points,["#8C559C","#127B8C","#74A05B"],label_offsets):
    ax.scatter(nmbe,cvrmse,s=78,color=colour,edgecolor="white",lw=.8,zorder=3)
    ax.annotate(label,(nmbe,cvrmse),xytext=offset,textcoords="offset points",
                ha="right" if offset[0]<0 else "left",va="bottom" if offset[1]>=0 else "top")
ax.set_xlim(0,4); ax.set_ylim(0,30)
ax.set_xlabel("Absolute normalized mean bias, |NMBE| (%)")
ax.set_ylabel("Relative daily electricity,\nCVRMSE (%)")
ax.text(.03,.06,"separate observed-load benchmarks",transform=ax.transAxes,color="#217A6B")
ax.grid(color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"b","Hong Kong campus relative-load comparisons")

ax=axs[1,0]
population_column="allocated_population__ssug_landuse_residential_gfa"
pred=pd.to_numeric(population[population_column],errors="raise").to_numpy(float)
obs=pd.to_numeric(population.official_population,errors="raise").to_numpy(float)
if len(obs)!=540 or not (np.isfinite(obs).all() and np.isfinite(pred).all()):
    raise ValueError("Figure 2c requires 540 finite estate observations from one allocation column")
norm_abs_error=100*np.abs(pred-obs).sum()/obs.sum()
norm_root_square_error=100*np.sqrt(np.square(pred-obs).sum()/np.square(obs).sum())
spearman=pd.Series(obs).rank(method="average").corr(pd.Series(pred).rank(method="average"))
assert np.isclose(norm_abs_error,4.215344,atol=1e-6)
assert np.isclose(norm_root_square_error,6.451348,atol=1e-6)
assert np.isclose(spearman,0.985895,atol=1e-6)
density=np.log10(np.maximum(obs,1))
sc=ax.scatter(obs,pred,c=density,cmap="viridis",s=26,marker="h",edgecolor="none",alpha=.88)
lim=max(np.nanmax(obs),np.nanmax(pred)); ax.plot([700,lim],[700,lim],"--",color="#D1495B",lw=1.2,label="1:1")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Official estate population"); ax.set_ylabel("Allocated building population")
ax.text(.04,.94,f"540 estates\nNormalized |error| {norm_abs_error:.2f}%\nNormalized root-square {norm_root_square_error:.2f}%\nSpearman {spearman:.3f}",transform=ax.transAxes,ha="left",va="top")
ax.legend(frameon=False,loc="lower right")
cb=fig.colorbar(sc,ax=ax,fraction=.040,pad=.012); cb.set_label("Estate size (log₁₀ population)")
panel_title(ax,"c","Population allocation against major estates")

ax=axs[1,1]
for i,row in heat.reset_index(drop=True).iterrows():
    ax.hlines(i-.09,row.observed_slope_p05,row.observed_slope_p95,color="#3D95A3",lw=5,alpha=.78)
    ax.scatter(row.observed_slope_median,i-.09,s=34,facecolor="white",edgecolor="#157080",zorder=3)
    ax.hlines(i+.09,row.model_slope_min,row.model_slope_max,color="#E07A67",lw=5,alpha=.78)
    ax.scatter(row.model_slope_median,i+.09,s=34,marker="D",facecolor="white",edgecolor="#D55742",zorder=3)
ax.set_yticks(range(len(heat)),["PolyU\nnon-residential","HKUST residence\nproxy","HKUST\nnon-residential"])
ax.set_xscale("log"); ax.set_xlabel("Relative daily electricity response per °C above balance point")
ax.legend(handles=[Line2D([],[],color="#3D95A3",lw=5,marker="o",markerfacecolor="white",label="Observed P05–P95; median"),
                   Line2D([],[],color="#E07A67",lw=5,marker="D",markerfacecolor="white",label="Model prototype range; median")],
          frameon=False,loc="upper left",ncol=1,bbox_to_anchor=(.01,.99))
ax.set_ylim(-.5,3.35)
ax.grid(axis="x",color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"d","Observed and simulated heat sensitivity")

ax=axs[2,0]
y=np.arange(len(bca)); ax.hlines(y,bca.bca_p05_eui,bca.bca_p95_eui,color="#A8B4BD",lw=6)
ax.vlines(bca.bca_median_eui,y-.14,y+.14,color="#334155",lw=1.5)
ax.scatter(bca.modeled_calibrated_eui_kwh_m2_year,y,s=45,marker="D",color="#278577",edgecolor="white",lw=.6,zorder=3)
eui_label_map={"com_office":"Office","com_retail":"Retail","com_accommodation":"Accommodation\n(hotel benchmark)","com_education":"Education\n(university benchmark)","com_human_health":"Healthcare\n(clinic benchmark)","com_mixed_use_component":"Mixed-use\ncomponent","com_other":"Other commercial\n(BCA mixed benchmark)"}
eui_labels=[eui_label_map.get(str(pid),str(label)) for pid,label in zip(bca.prototype_id,bca.bca_building_type)]
ax.set_yticks(y,eui_labels); ax.set_xlabel("Annual EUI (kWh m⁻² yr⁻¹)")
ax.grid(axis="x",color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"e","Hot-humid commercial EUI plausibility")

ax=axs[2,1]
order=["other_housing","subsidised_sale","public_housing","private_housing","commercial_sector_total"]
if energy.calibration_group.duplicated().any():
    raise ValueError("Figure 2f calibration keys must be unique")
missing=sorted(set(order)-set(energy.calibration_group))
if missing:
    raise ValueError(f"Figure 2f missing calibration groups: {missing}")
e=energy.set_index("calibration_group").loc[order].reset_index()
required_numeric=["official_total_kwh","calibrated_total_kwh","official_cooling_kwh"]
if len(e)!=5 or not np.isfinite(e[required_numeric].to_numpy(float)).all():
    raise ValueError("Figure 2f requires five complete finite calibration groups")
assert np.isclose(e.loc[e.calibration_group.eq("public_housing"),"official_total_kwh"].iloc[0]/1e9,3.242222,atol=1e-6)
y=np.arange(len(e)); official=e.official_total_kwh/1e9; calibrated=e.calibrated_total_kwh/1e9
ax.hlines(y,np.minimum(official,calibrated),np.maximum(official,calibrated),color="#A8B4BD",lw=2)
ax.scatter(official,y,s=58,color="#3C7188",label="Official target",zorder=3)
ax.scatter(calibrated,y,s=45,marker="D",color="#E07A5F",label="Calibrated model",zorder=4)
ax.scatter(official,y,s=22+75*(e.official_cooling_kwh/e.official_total_kwh),facecolor="none",edgecolor="#3C7188",lw=1)
wrapped={"other_housing":"Other\nhousing","subsidised_sale":"Subsidised\nsale","public_housing":"Public\nrental","private_housing":"Private\nhousing","commercial_sector_total":"Commercial\nsector total"}
ax.set_xscale("log"); ax.set_yticks(y,[wrapped[v] for v in e.calibration_group])
ax.set_xlabel("Annual electricity (TWh, log scale)")
ax.legend(frameon=False,loc="lower right"); ax.grid(axis="x",color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"f","EMSD annual calibration constraint")
finish(fig,"Figure_2_observational_constraints")


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# FIGURE 3 — territory centroid symbols using current event-integrated values.
# The former publication figure converted footprints to centroids and drew
# saturated 0.55–0.62-pt symbols.  That is retained here because most Hong Kong
# footprints are smaller than a final-pixel at the territory scale.  The source
# values, clipping rules and zero/missing states remain the current project outputs.
data=pd.read_csv(SD/"Figure3_2022_territory_buildings.csv")
fixed_path=SD/"Figure_building_geometry.parquet"
fixed=pd.read_parquet(fixed_path)
fixed["geometry_shape"]=list(from_wkb(fixed.geometry.to_numpy()))
data=data.merge(fixed[["BUILDINGSTRUCTUREID","geometry_shape"]],on="BUILDINGSTRUCTUREID",how="left",validate="one_to_one")
components=pd.read_parquet(STOCK/"building_component_map.parquet",columns=["BUILDINGSTRUCTUREID","population_reference_2021"])
population=components.groupby("BUILDINGSTRUCTUREID",as_index=False).population_reference_2021.sum()
data=data.merge(population,on="BUILDINGSTRUCTUREID",how="left",validate="one_to_one")
data=data[data.geometry_shape.notna()].copy()
centroids=gpd.GeoSeries(data.geometry_shape,crs=2326).centroid
data["centroid_x"]=centroids.x.to_numpy(); data["centroid_y"]=centroids.y.to_numpy()

land_path=BOUNDARIES/"housing_market_areas.geoparquet"
land=gpd.read_parquet(land_path).to_crs(2326) if land_path.exists() else None
xmin,xmax=data.centroid_x.min(),data.centroid_x.max(); ymin,ymax=data.centroid_y.min(),data.centroid_y.max()
xpad=(xmax-xmin)*.018; ypad=(ymax-ymin)*.018
extent=(xmin-xpad,xmax+xpad,ymin-ypad,ymax+ypad)

fig=plt.figure(figsize=(10.58,8.35),constrained_layout=False)
gs=fig.add_gridspec(2,6,left=.025,right=.915,bottom=.075,top=.960,
                    width_ratios=[4.75,.22,.90,.20,4.75,.22],wspace=.11,hspace=.22)
maps=[fig.add_subplot(gs[0,0]),fig.add_subplot(gs[0,4]),fig.add_subplot(gs[1,0]),fig.add_subplot(gs[1,4])]
side_type=fig.add_subplot(gs[0,1:3]); cbar_b=fig.add_subplot(gs[0,5]); cbar_c=fig.add_subplot(gs[1,1]); cbar_d=fig.add_subplot(gs[1,5])
for ax in maps:
    ax.set_facecolor("#F6F7F4")
    if land is not None:
        land.plot(ax=ax,color="#D5DBD7",edgecolor="#AEB7B4",linewidth=.10,zorder=0)
    ax.set_xlim(extent[0],extent[1]); ax.set_ylim(extent[2],extent[3])
    ax.set_aspect("equal",adjustable="box"); ax.set_axis_off()

ax=maps[0]
for category,colour in TYPE_COLORS.items():
    q=data.detailed_type.eq(category)
    ax.scatter(data.loc[q,"centroid_x"],data.loc[q,"centroid_y"],s=.62,c=colour,
               marker="o",linewidths=0,rasterized=True,zorder=2)
panel_title(ax,"a","Detailed Hong Kong building types")
side_type.axis("off")
side_type.legend(handles=[Patch(facecolor=c,edgecolor="none",label=k) for k,c in TYPE_COLORS.items()],
                 frameon=False,loc="center left",bbox_to_anchor=(0,.50),ncol=1,
                 handlelength=.85,handletextpad=.35,labelspacing=.30,borderaxespad=0)

ax=maps[1]
energy=data.E_HW_kwh.to_numpy(float); emax=np.nanpercentile(energy,99.7)
ecmap=plt.get_cmap("YlOrRd").copy(); ecmap.set_over("#7F0000")
sc=ax.scatter(data.centroid_x,data.centroid_y,c=energy,s=.62,cmap=ecmap,
              norm=colors.Normalize(0,emax,clip=False),linewidths=0,rasterized=True,zorder=2)
cb=fig.colorbar(sc,cax=cbar_b,extend="max",format=lambda x,pos:f"{x/1000:.0f}")
cb.set_label("Complete-event electricity\n(10³ kWh building⁻¹)",labelpad=7)
panel_title(ax,"b","Complete-event electricity, 19 July–2 August 2022")

ax=maps[2]
dh=data.absolute_dh32.to_numpy(float); dmax=np.nanpercentile(dh[np.isfinite(dh)],99.5)
dcmap=plt.get_cmap("YlGn").copy(); dcmap.set_over("#00441B")
sc=ax.scatter(data.centroid_x,data.centroid_y,c=dh,s=.62,cmap=dcmap,
              norm=colors.Normalize(0,dmax,clip=False),linewidths=0,rasterized=True,zorder=2)
cb=fig.colorbar(sc,cax=cbar_c,extend="max"); cb.set_label("Outage DH32\n(K h)",labelpad=7)
panel_title(ax,"c","Outage overheating\n24 Jul 15:00–25 Jul 15:00 HKT")

ax=maps[3]
burden=data.population_degree_hours.to_numpy(float)
has_residential_population=pd.to_numeric(data.population_reference_2021,errors="coerce").fillna(0).gt(0)
not_applicable=~has_residential_population
true_zero=np.isfinite(burden)&(burden==0)&has_residential_population
missing=~np.isfinite(burden)&has_residential_population
positive=np.isfinite(burden)&(burden>0)&has_residential_population
ax.scatter(data.loc[not_applicable,"centroid_x"],data.loc[not_applicable,"centroid_y"],s=.72,c="#707879",linewidths=0,rasterized=True,zorder=1)
ax.scatter(data.loc[true_zero,"centroid_x"],data.loc[true_zero,"centroid_y"],s=.82,c="#FFF2A8",edgecolors="#5E5E42",linewidths=.05,rasterized=True,zorder=2)
if missing.any():
    ax.scatter(data.loc[missing,"centroid_x"],data.loc[missing,"centroid_y"],s=.82,c="#252525",linewidths=0,rasterized=True,zorder=2)
bmin=max(np.nanpercentile(burden[positive],1),1e-3); bmax=np.nanpercentile(burden[positive],99.5)
bcmap=plt.get_cmap("PuRd").copy(); bcmap.set_over("#67001F")
sc=ax.scatter(data.loc[positive,"centroid_x"],data.loc[positive,"centroid_y"],c=burden[positive],s=.62,
              cmap=bcmap,norm=colors.LogNorm(bmin,bmax,clip=False),linewidths=0,rasterized=True,zorder=3)
cb=fig.colorbar(sc,cax=cbar_d,extend="max"); cb.set_label("Fixed-population heat burden\n(person K h building⁻¹)",labelpad=7)
panel_title(ax,"d","Residential population-weighted heat burden")
state_handles=[Patch(facecolor="#FFF2A8",edgecolor="#5E5E42",label="True zero burden"),
               Patch(facecolor="#707879",edgecolor="none",label="No residential population / not applicable")]
if missing.any(): state_handles.append(Patch(facecolor="#252525",edgecolor="none",label="Missing"))
fig.legend(handles=state_handles,frameon=False,loc="lower right",bbox_to_anchor=(.965,.012),ncol=len(state_handles))
finish(fig,"Figure_3_territory_spatial_patterns")


In [ ]:
from pathlib import Path
from zoneinfo import ZoneInfo
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# FIGURE 4 — neighbourhood peak-demand maps, outage +12 h thermal state, static BERI and continuous demand.
regions=["Ho Man Tin","Mong Kok","Tsim Sha Tsui"]
slugs={"Ho Man Tin":"Ho_Man_Tin","Mong Kok":"Mong_Kok","Tsim Sha Tsui":"Tsim_Sha_Tsui"}
HKT=ZoneInfo("Asia/Hong_Kong")
event_start=pd.Timestamp("2022-07-19 00:00",tz=HKT)
event_end=pd.Timestamp("2022-08-03 00:00",tz=HKT)
outage_start=pd.Timestamp("2022-07-24 15:00",tz=HKT)
restoration=pd.Timestamp("2022-07-25 15:00",tz=HKT)
thermal_time=pd.Timestamp("2022-07-25 03:00",tz=HKT)

fixed=pd.read_parquet(SD/"Figure_building_geometry.parquet")
fixed["geometry_shape"]=list(from_wkb(fixed.geometry.to_numpy()))
types=pd.read_csv(SD/"Figure3_2022_territory_buildings.csv",usecols=["BUILDINGSTRUCTUREID","detailed_type"])
curves=pd.read_csv(SD/"Figure4_regional_load_intensity_15min.csv")
curves["timestamp_hkt"]=pd.to_datetime(curves.timestamp_hkt,utc=True).dt.tz_convert(HKT)
times=pd.read_csv(SD/"Figure4_regional_map_times.csv")
times["load_map_timestamp_hkt"]=pd.to_datetime(times.load_map_timestamp_hkt,utc=True).dt.tz_convert(HKT)

region_frames=[]
for region in regions:
    q=pd.read_csv(SD/f"Figure4_{slugs[region]}_dynamic_buildings.csv")
    q["load_map_timestamp_hkt"]=pd.to_datetime(q.load_map_timestamp_hkt,utc=True).dt.tz_convert(HKT)
    q=q.merge(types,on="BUILDINGSTRUCTUREID",how="left",validate="one_to_one")
    q=q.merge(fixed[["BUILDINGSTRUCTUREID","geometry_shape"]],on="BUILDINGSTRUCTUREID",how="left",validate="one_to_one")
    region_frames.append(q)
all_buildings=pd.concat(region_frames,ignore_index=True)

load_values=all_buildings.load_intensity_w_m2.to_numpy(float)
load_norm=colors.Normalize(0,np.nanpercentile(load_values,99.2),clip=False)
load_cmap=plt.get_cmap("YlOrRd").copy(); load_cmap.set_over("#7F0000")
thermal_values=all_buildings.thermal_margin_to_32c.to_numpy(float)
tlo=min(-5,np.nanpercentile(thermal_values,1)); thi=max(2,np.nanpercentile(thermal_values,99))
thermal_norm=colors.TwoSlopeNorm(vmin=tlo,vcenter=0,vmax=thi)
thermal_cmap=plt.get_cmap("RdYlGn")
thermal_state_colors={"partial_support":"#B8C0C2","no_eligible_object":"#707879","missing":"#252525"}
beri_norm=colors.Normalize(0,100); beri_cmap=plt.get_cmap("RdYlGn")

road_dir=ROADS
road_frames=[gpd.read_file(p) for p in sorted(road_dir.glob("*_simplified5m.geojson"))]
roads=gpd.GeoDataFrame(pd.concat(road_frames,ignore_index=True),crs=road_frames[0].crs).to_crs(2326) if road_frames else None

fig=plt.figure(figsize=(15.70,11.70),constrained_layout=False)
outer=fig.add_gridspec(3,1,left=.042,right=.968,bottom=.205,top=.975,hspace=.36)
letters=list("abcdefghijklmno")
for row,region in enumerate(regions):
    block=outer[row].subgridspec(2,1,height_ratios=[5.45,1.12],hspace=.36)
    top=block[0].subgridspec(1,7,width_ratios=[4.95,4.95,.18,4.95,.18,4.95,.18],wspace=.26)
    maps=[fig.add_subplot(top[0,0]),fig.add_subplot(top[0,1]),fig.add_subplot(top[0,3]),fig.add_subplot(top[0,5])]
    bars=[fig.add_subplot(top[0,2]),fig.add_subplot(top[0,4]),fig.add_subplot(top[0,6])]
    curve=fig.add_subplot(block[1])
    local=region_frames[row].copy()
    geoms=gpd.GeoSeries(local.geometry_shape,crs=2326)
    xmin,ymin,xmax,ymax=geoms.total_bounds; cx=(xmin+xmax)/2; cy=(ymin+ymax)/2; span=max(xmax-xmin,ymax-ymin)*1.035
    bounds=(cx-span/2,cy-span/2,cx+span/2,cy+span/2)
    for ax in maps:
        ax.set_facecolor("white"); ax.set_xlim(bounds[0],bounds[2]); ax.set_ylim(bounds[1],bounds[3])
        ax.set_aspect("equal",adjustable="box"); ax.set_box_aspect(1); ax.set_axis_off()
    polygon_layer(maps[0],local.geometry_shape,local.detailed_type,categories=TYPE_COLORS,edge="none",linewidth=0)
    load=polygon_layer(maps[1],local.geometry_shape,local.load_intensity_w_m2,cmap=load_cmap,norm=load_norm,edge="none",linewidth=0)
    thermal_state=local.thermal_support_state.fillna("missing")
    thermal_ok=thermal_state.eq("complete_evaluable")&local.thermal_margin_to_32c.notna()
    for support_state,support_colour in thermal_state_colors.items():
        state_mask=thermal_state.eq(support_state)
        if state_mask.any(): polygon_layer(maps[2],local.loc[state_mask,"geometry_shape"],[support_state]*int(state_mask.sum()),categories={support_state:support_colour},edge="none",linewidth=0)
    thermal=polygon_layer(maps[2],local.loc[thermal_ok,"geometry_shape"],local.loc[thermal_ok,"thermal_margin_to_32c"],cmap=thermal_cmap,norm=thermal_norm,edge="none",linewidth=0)
    beri_ok=local.BERI_lower.notna()
    if (~beri_ok).any(): polygon_layer(maps[3],local.loc[~beri_ok,"geometry_shape"],["na"]*int((~beri_ok).sum()),categories={"na":"#707879"},edge="none",linewidth=0)
    beri=polygon_layer(maps[3],local.loc[beri_ok,"geometry_shape"],local.loc[beri_ok,"BERI_lower"],cmap=beri_cmap,norm=beri_norm,edge="none",linewidth=0)
    for ax in maps:
        if roads is not None:
            subset=roads.cx[bounds[0]:bounds[2],bounds[1]:bounds[3]]
            if not subset.empty: subset.plot(ax=ax,color="#5F6868",linewidth=.30,alpha=.50,zorder=8)
        ax.set_xlim(bounds[0],bounds[2]); ax.set_ylim(bounds[1],bounds[3]); ax.set_box_aspect(1)
        bar_len=500 if span>1200 else 250; x0=bounds[0]+span*.055; y0=bounds[1]+span*.045
        ax.plot([x0,x0+bar_len],[y0,y0],color="black",lw=2,zorder=9)
        ax.text(x0+bar_len/2,y0+span*.017,f"{bar_len} m",ha="center",va="bottom",fontsize=11,zorder=9)
    titles=[f"{region} · building type","Load intensity at local peak","Thermal margin · outage +12 h","Reference-suite BERI"]
    base_letter=row*5
    for offset,(ax,title) in enumerate(zip(maps,titles)): panel_title(ax,letters[base_letter+offset],title)
    cb=fig.colorbar(load,cax=bars[0],extend="max"); cb.set_label("Load intensity (W m⁻²)",labelpad=4)
    cb=fig.colorbar(thermal,cax=bars[1],extend="both"); cb.set_label("32 °C − warmest eligible zone (°C)",labelpad=4)
    cb=fig.colorbar(beri,cax=bars[2]); cb.set_label("BERI lower endpoint (0–100)",labelpad=4)

    q=curves[curves.study_region.eq(region)].sort_values("timestamp_hkt")
    peak=times[times.study_region.eq(region)].iloc[0].load_map_timestamp_hkt
    peak_value=q.loc[q.timestamp_hkt.eq(peak),"area_weighted_load_intensity_w_m2"].iloc[0]
    curve.plot(q.timestamp_hkt,q.area_weighted_load_intensity_w_m2,color="#2F75B5",lw=1.30)
    curve.axvspan(outage_start,restoration,color="#7A5195",alpha=.11)
    curve.axvline(peak,color="#E69F00",lw=1.25)
    curve.scatter([peak],[peak_value],s=38,color="#E69F00",edgecolor="white",lw=.6,zorder=5)
    curve.axvline(thermal_time,color="#009E73",lw=1.15,ls="--")
    curve.set_xlim(event_start,event_end); curve.set_ylim(0,q.area_weighted_load_intensity_w_m2.max()*1.30)
    curve.set_ylabel("Load intensity\n(W m⁻²)",rotation=90,labelpad=6)
    curve.grid(axis="y",color="#D9DDE0",lw=.5,alpha=.55)
    curve.xaxis.set_major_locator(mdates.DayLocator(interval=3,tz=HKT)); curve.xaxis.set_major_formatter(mdates.DateFormatter("%d %b",tz=HKT))
    curve.set_xlabel("Event time (HKT)",labelpad=2)
    curve.text(.995,1.055,f"n={len(local):,}  |  load map {peak.strftime('%d %b %H:%M')}  |  thermal map {thermal_time.strftime('%d %b %H:%M')} HKT",transform=curve.transAxes,ha="right",va="center",clip_on=False)
    panel_title(curve,letters[base_letter+4],f"{region} · area-weighted regional load intensity")

type_handles=[Patch(facecolor=c,edgecolor="none",label=k) for k,c in TYPE_COLORS.items()]
fig.legend(handles=type_handles,frameon=False,loc="lower center",ncol=7,bbox_to_anchor=(.50,.105),columnspacing=.62,handlelength=.82,handletextpad=.28,labelspacing=.28)
support_handles=[Patch(facecolor=thermal_state_colors["partial_support"],edgecolor="none",label="Thermal map: partial evaluation support"),
                 Patch(facecolor=thermal_state_colors["no_eligible_object"],edgecolor="none",label="Thermal map: no eligible object")]
fig.legend(handles=support_handles,frameon=False,loc="lower center",ncol=2,bbox_to_anchor=(.50,.068),columnspacing=1.10,handlelength=.82,handletextpad=.34)
process_handles=[Line2D([],[],color="#2F75B5",lw=1.30,label="HW_UNL 15-min area-weighted demand intensity"),
                 Line2D([],[],color="#E69F00",marker="o",lw=1.25,label="Local load-map time"),
                 Line2D([],[],color="#009E73",ls="--",lw=1.15,label="O24 thermal-map time · outage +12 h"),
                 Patch(facecolor="#7A5195",alpha=.11,label="O24 outage window · 24 Jul 15:00–25 Jul 15:00")]
fig.legend(handles=process_handles,frameon=False,loc="lower center",ncol=4,bbox_to_anchor=(.50,.030),columnspacing=.90,handletextpad=.38)
fig.text(.50,.010,"19 July–2 August 2022 · interval-ending 15-min means · road context: Hong Kong Transport Department / CSDI",ha="center",va="bottom",fontsize=11)
finish(fig,"Figure_4_neighbourhood_contrasts")


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# FIGURE 5 — eight-event synthesis; fold distributions are dot-and-range displays.
classes=pd.read_csv(SD/"Figure5_class.csv")
city=pd.read_csv(SD/"Figure5_city.csv")
rank=pd.read_csv(SD/"Figure5_rank.csv")
capture=pd.read_csv(SD/"Figure5_capture.csv")
lock_rank=pd.read_csv(ANALYSIS/"first_look_rank_summary.csv")
lock_capture=pd.read_csv(ANALYSIS/"first_look_top10_capture_summary.csv")

fig,axs=plt.subplots(3,2,figsize=(10.58,11.15),constrained_layout=False)
fig.subplots_adjust(left=.145,right=.965,bottom=.070,top=.970,wspace=.40,hspace=.50)
class_order=["Residential","Mixed-use","Commercial"]
outcomes=["cooling_shortfall_kwh","absolute_dh32","extra_dh26","population_degree_hours","recovery_hours"]
outcome_labels=["Cooling shortfall","Outage DH32","Added DH26","Population heat","Recovery"]
outcome_colours=["#2F75B5","#D55E00","#009E73","#9C4F96","#4D4D4D"]
outcome_markers=["o","s","D","^","P"]

ax=axs[0,0]
values=[classes.loc[classes.main_class.eq(k),"median_relative_net_delta_pct"].to_numpy(float) for k in class_order]
vp=ax.violinplot(values,positions=np.arange(3),widths=.62,showmedians=False,showextrema=False)
for body,k in zip(vp["bodies"],class_order): body.set_facecolor(CLASS_COLORS[k]); body.set_edgecolor("none"); body.set_alpha(.52)
for i,k in enumerate(class_order):
    q=classes[classes.main_class.eq(k)].sort_values("year"); jitter=np.linspace(-.10,.10,len(q))
    ax.scatter(i+jitter,q.median_relative_net_delta_pct,s=34,color=CLASS_COLORS[k],edgecolor="white",lw=.55,zorder=3)
    ax.hlines(q.median_relative_net_delta_pct.median(),i-.22,i+.22,color="#202020",lw=1.5,zorder=4)
ax.set_xticks(range(3),class_order); ax.set_ylabel("Event-median building electricity\nincrease (%)")
ax.grid(axis="y",color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"a","Building-use electricity sensitivity across eight events")

ax=axs[0,1]; city=city.sort_values("year"); x=np.arange(len(city))
ax.bar(x,city.net_delta_gwh,color="#547B96",alpha=.92)
ax.set_ylabel("Heatwave–reference event electricity (GWh)",color="#335B73")
ax.set_xticks(x,city.year.astype(str),rotation=45); ax.set_xlabel("Event year"); ax.grid(axis="y",color="#D9DDE0",lw=.5,alpha=.55)
ax2=ax.twinx(); ax2.plot(x,city.same_time_difference_at_hw_peak_gw,color="#D1495B",marker="D",lw=1.4)
ax2.plot(x,city.difference_of_separate_maxima_gw,color="#D1495B",marker="s",markerfacecolor="white",lw=.9,alpha=.75)
ax2.axhline(0,color="#666",lw=.6); ax2.set_ylabel("Peak difference (GW)",color="#D1495B")
ax.set_ylim(min(0,city.net_delta_gwh.min()*1.10),city.net_delta_gwh.max()*1.58)
ax2.set_ylim(min(-.7,city.difference_of_separate_maxima_gw.min()*1.30),city.same_time_difference_at_hw_peak_gw.max()*1.58)
handles=[Patch(facecolor="#547B96",label="Event electricity"),Line2D([],[],color="#D1495B",marker="D",label="Same-time peak"),Line2D([],[],color="#D1495B",marker="s",markerfacecolor="white",label="Difference of maxima")]
ax.legend(handles=handles,frameon=False,loc="upper left",bbox_to_anchor=(0,.995),ncol=1)
panel_title(ax,"b","Event electricity and territory-wide peaks")

ax=axs[1,0]
labels=["Cooling\nshortfall","Outage\nDH32","Added\nDH26","Population\nheat","Recovery"]
for j,(metric,label) in enumerate(zip(outcomes,labels)):
    rr=lock_rank[(lock_rank.method.eq("BERI"))&lock_rank.outcome.eq(metric)].iloc[0]
    cc=lock_capture[(lock_capture.method.eq("BERI"))&lock_capture.outcome.eq(metric)].iloc[0]
    ax.plot([rr.min_spearman,rr.max_spearman],[j,j],color="#2C7BB6",lw=2.2)
    ax.scatter(rr.median_spearman,j,s=56+420*cc.median_capture,color=plt.cm.viridis(cc.median_capture/.20),edgecolor="white",lw=.7,zorder=3)
    ax.text(rr.max_spearman+.025,j,f"capture {cc.median_capture*100:.1f}%",va="center",ha="left")
ax.axvline(0,color="#777",lw=.7); ax.set_yticks(range(5),labels); ax.invert_yaxis(); ax.set_xlim(-.10,1.30)
ax.set_xlabel("Spearman rank agreement\n(two independent events)")
ax.grid(axis="x",color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"c","Independent-event first-look screening")

method_order=["BERI","stage_N","stage_C","stage_O","stage_R","building_type","detailed_use_floor","construction_era"]
method_labels=["BERI","Stage N","Stage C","Stage O","Stage R","Building type","Detailed use + floor","Construction era"]
offsets=np.linspace(-.22,.22,len(outcomes))
ax=axs[1,1]
for j,(outcome,label,colour,marker,off) in enumerate(zip(outcomes,outcome_labels,outcome_colours,outcome_markers,offsets)):
    q=rank[rank.outcome.eq(outcome)].set_index("method").reindex(method_order)
    y=np.arange(len(method_order))+off
    ax.hlines(y,q.min_spearman,q.max_spearman,color=colour,lw=1.25,alpha=.72)
    ax.scatter(q.median_spearman,y,s=31,color=colour,marker=marker,edgecolor="white",lw=.4,zorder=3,label=label)
ax.axvline(0,color="#777",lw=.7); ax.set_yticks(range(len(method_order)),method_labels)
ax.set_ylim(len(method_order)-.55,-3.65); ax.set_xlim(-.12,1.03)
ax.set_xlabel("Spearman ρ: median and min–max across eight LOEO folds")
ax.grid(axis="x",color="#D9DDE0",lw=.5,alpha=.55)
ax.legend(frameon=False,loc="upper right",bbox_to_anchor=(.995,.995),ncol=2,columnspacing=.7,handletextpad=.3)
panel_title(ax,"d","Event-held-out rank agreement")

ax=axs[2,0]
for yi,k in enumerate(class_order):
    q=classes[classes.main_class.eq(k)].set_index("year").reindex(EVENT_YEARS); vals=q.median_absolute_dh32_Kh.to_numpy(float)
    ax.scatter(np.arange(8),np.full(8,yi),s=40+8*np.sqrt(np.maximum(vals,0)),c=vals,cmap="magma_r",vmin=0,vmax=max(1,np.nanmax(classes.median_absolute_dh32_Kh)),edgecolor="white",lw=.55)
    for xi,v in enumerate(vals): ax.text(xi,yi-.25,f"{v:.1f}",ha="center",va="top")
ax.set_xticks(range(8),EVENT_YEARS,rotation=45); ax.set_yticks(range(3),class_order); ax.set_ylim(2.55,-.55)
ax.set_xlabel("Event year"); ax.set_ylabel("Building\nuse")
panel_title(ax,"e","24-h outage DH32 (K h): category medians")

ax=axs[2,1]
for j,(outcome,label,colour,marker,off) in enumerate(zip(outcomes,outcome_labels,outcome_colours,outcome_markers,offsets)):
    q=capture[capture.outcome.eq(outcome)].set_index("method").reindex(method_order)
    y=np.arange(len(method_order))+off
    ax.hlines(y,q.min_top10_capture*100,q.max_top10_capture*100,color=colour,lw=1.25,alpha=.72)
    ax.scatter(q.median_top10_capture*100,y,s=31,color=colour,marker=marker,edgecolor="white",lw=.4,zorder=3,label=label)
ax.axvline(10,color="#777",lw=.7,ls="--"); ax.set_yticks(range(len(method_order)),method_labels)
ax.set_ylim(len(method_order)-.55,-3.65); ax.set_xlim(0,max(48,capture.max_top10_capture.max()*100*1.08))
ax.set_xlabel("Top-decile burden captured (%): median and min–max")
ax.grid(axis="x",color="#D9DDE0",lw=.5,alpha=.55)
ax.legend(frameon=False,loc="upper right",bbox_to_anchor=(.995,.995),ncol=2,columnspacing=.7,handletextpad=.3)
panel_title(ax,"f","Event-held-out burden capture")
finish(fig,"Figure_5_event_and_use_dependent_vulnerability")


In [ ]:
from pathlib import Path
from zoneinfo import ZoneInfo
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# FIGURE 6 — event timing and controlled mechanisms in five panels.
effects=pd.read_csv(SD/"Figure6_controlled_pair_effects.csv")
trajectory=pd.read_csv(SD/"Figure6c_2022_recovery_trajectories.csv",parse_dates=["timestamp_hkt"])
HKT=ZoneInfo("Asia/Hong_Kong")
city_curve=pd.read_csv(SD/"Figure6a_2009_city_trajectory.csv")
city_curve["timestamp_hkt"]=pd.to_datetime(city_curve.timestamp_hkt,utc=True).dt.tz_convert(HKT)
trajectory["timestamp_hkt"]=pd.to_datetime(trajectory.timestamp_hkt,utc=True).dt.tz_convert(HKT)
known_input=pd.Timestamp("2022-07-19 00:15",tz=HKT)
assert mdates.num2date(mdates.date2num(known_input.to_pydatetime()),tz=HKT).strftime("%d %b %H:%M")=="19 Jul 00:15"
order=["Top exposure","Mass −15%","Mass +15%","Capacity −15%","Capacity +15%","AC 1 h earlier","AC 1 h later"]
effects=effects[effects.display.isin(order)].copy()
factor_colours={"roof_top_exposure":"#D55E00","thermal_mass":"#009E73","nominal_cooling_capacity":"#0072B2","AC_operating_time":"#9C4F96"}
fig=plt.figure(figsize=(10.58,8.95),constrained_layout=False)
outer=fig.add_gridspec(2,2,left=.155,right=.970,bottom=.075,top=.955,wspace=.30,hspace=.42)

left=outer[0,0].subgridspec(2,1,height_ratios=[2.45,1.15],hspace=.42)
ax=fig.add_subplot(left[0,0]); city_curve=city_curve.sort_values("timestamp_hkt")
ax.plot(city_curve.timestamp_hkt,city_curve.power_kw_hw/1e6,color="#D1495B",lw=1.4,label="Heatwave weather")
ax.plot(city_curve.timestamp_hkt,city_curve.power_kw_cf/1e6,color="#526779",lw=1.3,label="Matched reference")
ax.fill_between(city_curve.timestamp_hkt,city_curve.power_kw_cf/1e6,city_curve.power_kw_hw/1e6,
                where=city_curve.power_kw_hw.ge(city_curve.power_kw_cf),color="#E8B474",alpha=.32,label="Positive heatwave increment")
ax.set_ylabel("Hong Kong building load (GW)"); ax.grid(color="#D9DDE0",lw=.5,alpha=.55)
ax.set_ylim(city_curve[["power_kw_hw","power_kw_cf"]].to_numpy(float).min()/1e6*.94,20.5)
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2,tz=HKT)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b",tz=HKT))
ax.legend(frameon=False,loc="upper left",bbox_to_anchor=(.01,.995),ncol=1)
panel_title(ax,"a","Heatwave and reference demand")

bx=fig.add_subplot(left[1,0])
hw_peak=city_curve.loc[city_curve.power_kw_hw.idxmax()]; cf_peak=city_curve.loc[city_curve.power_kw_cf.idxmax()]
same=(hw_peak.power_kw_hw-hw_peak.power_kw_cf)/1e3; separate=(hw_peak.power_kw_hw-cf_peak.power_kw_cf)/1e3
terms=[("Same-time\ndifference",same,"#D1495B"),("Difference of\nmaxima",separate,"#526779")]
for yi,(label,val,col) in enumerate(terms):
    bx.hlines(yi,0,val,color=col,lw=5,alpha=.75); bx.scatter(val,yi,marker="D",s=40,color=col,edgecolor="white",zorder=3)
bx.axvline(0,color="#666",lw=.7); bx.set_yticks([0,1],[t[0] for t in terms]); bx.set_xlabel("Peak-difference term (MW)")
bx.grid(axis="x",color="#D9DDE0",lw=.5,alpha=.55); panel_title(bx,"b","Two non-equivalent peak definitions")

ax=fig.add_subplot(outer[0,1]); restoration=pd.Timestamp("2022-07-25 15:00:00",tz=HKT)
trajectory["hours"]=(trajectory.timestamp_hkt-restoration).dt.total_seconds()/3600
trajectory_colours={("res_old_lowrise_tenement","middle"):"#2C7BB6",("res_old_lowrise_tenement","top"):"#D95F02",
                    ("res_private_highrise","middle"):"#1B9E77",("res_private_highrise","top"):"#8E44AD"}
for (prototype,floor),q in trajectory.groupby(["prototype_id","floor_position"]):
    label=("Old low-rise" if "old_lowrise" in prototype else "Private high-rise")+f" · {floor}"
    ax.plot(q.hours,q.temperature_c,color=trajectory_colours[(prototype,floor)],lw=1.65,label=label)
ax.axvline(0,color="#202020",lw=1.0,ls="-.",label="Power restored · 25 Jul 15:00")
ax.axhline(32,color="#D1495B",ls=":",lw=1,label="32 °C comparison threshold")
ax.axhline(26,color="#2C7FB8",ls="--",lw=1,label="26 °C recovery target")
ax.set_xlabel("Hours after restoration at 25 July 2022 15:00 HKT"); ax.set_ylabel("Warmest conditioned-zone temperature (°C)")
ax.set_ylim(min(24,trajectory.temperature_c.min()-1),55.0)
ax.legend(frameon=False,loc="upper right",bbox_to_anchor=(.99,.995),ncol=1); ax.grid(color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"c","Post-restoration indoor-temperature trajectories")

ax=fig.add_subplot(outer[1,0]); rng=np.random.default_rng(42)
for i,label in enumerate(order):
    q=effects[effects.display.eq(label)]; factor=q.factor.iloc[0]; colour=factor_colours[factor]
    y=np.full(len(q),i)+rng.uniform(-.15,.15,len(q)); x=q.effect_shortfall_ref_kwh_thermal.to_numpy(float)
    ax.scatter(x,y,s=34,color=colour,alpha=.62,edgecolor="white",lw=.4,rasterized=True)
    if len(q)!=15:
        raise ValueError(f"Expected 15 paired effects for {label}, found {len(q)}")
    med=np.median(x); q10,q90=np.quantile(x,[.1,.9]); ax.plot([q10,q90],[i,i],color="#202020",lw=1.3,zorder=4)
    ax.scatter(med,i,s=42,marker="D",facecolor="white",edgecolor="#202020",lw=1.0,zorder=5)
ax.axvline(0,color="#666",lw=.8); ax.set_yticks(range(len(order)),order); ax.invert_yaxis()
ax.set_xlabel("Treatment – reference C70 shortfall (kWh thermal)"); ax.grid(axis="x",color="#D9DDE0",lw=.5,alpha=.55)
panel_title(ax,"d","Paired effects on cooling-service shortfall")

ax=fig.add_subplot(outer[1,1]); event_markers={2018:"o",2022:"s",2023:"D"}
for (factor,year),q in effects.groupby(["factor","year"]):
    ax.scatter(q.effect_shortfall_ref_kwh_thermal,q.effect_absolute_occupied_degree_hours_above_32c,
               s=40,marker=event_markers.get(int(year),"o"),color=factor_colours[factor],alpha=.62,
               edgecolor="white",lw=.45,rasterized=True)
ax.axhline(0,color="#666",lw=.7); ax.axvline(0,color="#666",lw=.7)
ax.set_xlabel("Δ C70 shortfall (kWh thermal)"); ax.set_ylabel("Δ outage DH32 (K h)")
factor_label={"roof_top_exposure":"Roof-top exposure","thermal_mass":"Thermal mass","nominal_cooling_capacity":"Cooling capacity","AC_operating_time":"AC schedule phase shift"}
factor_handles=[Line2D([],[],marker="o",ls="",color=c,label=factor_label[f]) for f,c in factor_colours.items()]
event_handles=[Line2D([],[],marker=m,ls="",color="#444",label=str(y)) for y,m in event_markers.items()]
ylo,yhi=ax.get_ylim(); ax.set_ylim(ylo,max(24.0,yhi+(yhi-ylo)*.28))
year_legend=ax.legend(handles=event_handles,frameon=False,loc="upper left",bbox_to_anchor=(.01,.995),ncol=1,title="Event year")
ax.add_artist(year_legend)
ax.legend(handles=factor_handles,frameon=False,loc="upper right",bbox_to_anchor=(.99,.995),ncol=1,title="Controlled change")
ax.grid(color="#D9DDE0",lw=.5,alpha=.55); panel_title(ax,"e","Mechanism-specific service and overheating effects")
finish(fig,"Figure_6_physical_and_operational_controls")


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S1 — full editable plotting cell.
tmax = pd.read_csv(HAZ / 'wrf_tmax_distributional_spatiotemporal_holdout.csv')
tmin = pd.read_csv(HAZ / 'wrf_tmin_distributional_spatiotemporal_holdout.csv')
fig, axs = plt.subplots(1, 2, figsize=(6.7, 3.55), constrained_layout=True)
for ax, d, title in [(axs[0], tmax, 'Daily maximum temperature'), (axs[1], tmin, 'Daily minimum temperature')]:
    ax.scatter(d.raw_quantile_mae_c, d.corrected_quantile_mae_c, s=8, alpha=0.45)
    lim = max(d.raw_quantile_mae_c.max(), d.corrected_quantile_mae_c.max())
    ax.plot([0, lim], [0, lim], color='#777', lw=0.8)
    ax.set(xlabel='Raw quantile MAE (°C)', ylabel='Corrected quantile MAE (°C)', title=title)
finish(fig, 'Supplementary_Figure_S1_weather_correction', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S2 — eight selected events at a common extent and scale.
links=pd.read_parquet(STOCK/"building_spatial_links.parquet")
links=links[links.study_region.eq("Hong Kong total")][["BUILDINGSTRUCTUREID","centroid_x","centroid_y"]]
key='population'; col='population_degree_hours'; den=None; label='Residential burden (log₁₀ person K h)'
vmin=-2; vmax=4; cmap='plasma'
xmin,xmax=links.centroid_x.min(),links.centroid_x.max(); ymin,ymax=links.centroid_y.min(),links.centroid_y.max()
xpad,ypad=(xmax-xmin)*.012,(ymax-ymin)*.012
fig,axs=plt.subplots(4,2,figsize=(10.58,13.10),constrained_layout=False)
fig.subplots_adjust(left=.020,right=.980,bottom=.070,top=.975,wspace=.025,hspace=.090)
cm=plt.get_cmap(cmap).copy(); cm.set_over(cm(1.0)); norm=colors.Normalize(vmin,vmax,clip=False)
sc=None
land_path=BOUNDARIES/'housing_market_areas.geoparquet'
land=gpd.read_parquet(land_path).to_crs(2326) if land_path.exists() else None
for idx,(ax,event,year) in enumerate(zip(axs.ravel(),EVENTS,YEARS)):
    ax.set_facecolor('#F6F7F4')
    if land is not None:
        land.plot(ax=ax,color='#D5DBD7',edgecolor='#AEB7B4',linewidth=.10,zorder=0)
    columns=["BUILDINGSTRUCTUREID",col]+([den] if den else [])
    d=pd.read_parquet(EVENTS_DIR/f"{event}.parquet",columns=columns)
    d=links.merge(d,on="BUILDINGSTRUCTUREID",validate="one_to_one")
    val=np.where(d[col]>0,np.log10(d[col]),np.nan)

    missing=~np.isfinite(d[col].to_numpy(float)); zero=np.isfinite(d[col].to_numpy(float))&d[col].eq(0).to_numpy()
    ax.scatter(d.loc[zero,"centroid_x"],d.loc[zero,"centroid_y"],s=.62,c="#F2F2E9",linewidths=0,rasterized=True,zorder=1)
    ax.scatter(d.loc[missing,"centroid_x"],d.loc[missing,"centroid_y"],s=.62,c="#707777",linewidths=0,rasterized=True,zorder=2)
    positive=np.isfinite(val)
    sc=ax.scatter(d.loc[positive,"centroid_x"],d.loc[positive,"centroid_y"],c=val[positive],s=.62,
                  cmap=cm,norm=norm,rasterized=True,linewidths=0,zorder=3)
    ax.set_xlim(xmin-xpad,xmax+xpad); ax.set_ylim(ymin-ypad,ymax+ypad)
    ax.set_aspect("equal",adjustable="box"); ax.set_axis_off(); panel_title(ax,chr(97+idx),str(year))
cax=fig.add_axes([.30,.038,.40,.014]); cb=fig.colorbar(sc,cax=cax,orientation="horizontal",extend="max"); cb.set_label(label)

fig.legend(handles=[Patch(facecolor="#F2F2E9",edgecolor="#777",label="Valid zero burden"),Patch(facecolor="#707777",edgecolor="none",label="Missing")],
           frameon=False,loc="lower right",bbox_to_anchor=(.985,.010),ncol=1)
finish(fig,'Supplementary_Figure_S2_eight_event_population_maps',supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S3 — complete-event electricity for all eight selected events at a common extent and scale.
data=pd.read_parquet(SD/"Supplementary_Figure_S3_eight_event_complete_electricity.parquet")
land_path=BOUNDARIES/"housing_market_areas.geoparquet"
land=gpd.read_parquet(land_path).to_crs(2326) if land_path.exists() else None
xmin,xmax=data.centroid_x.min(),data.centroid_x.max(); ymin,ymax=data.centroid_y.min(),data.centroid_y.max()
xpad=(xmax-xmin)*.018; ypad=(ymax-ymin)*.018; extent=(xmin-xpad,xmax+xpad,ymin-ypad,ymax+ypad)
positive=data.loc[data.E_HW_kwh.gt(0),"E_HW_kwh"].to_numpy(float)
norm=colors.LogNorm(max(np.nanpercentile(positive,.5),1e-3),np.nanpercentile(positive,99.7),clip=False)
cmap=plt.get_cmap("YlOrRd").copy(); cmap.set_over("#7F0000")
years=[2005,2009,2012,2014,2017,2018,2022,2023]
fig,axs=plt.subplots(4,2,figsize=(8.25,10.45),constrained_layout=False)
fig.subplots_adjust(left=.025,right=.900,bottom=.055,top=.975,wspace=.020,hspace=.105)
letters=list("abcdefgh")
artist=None
for ax,year,letter in zip(axs.flat,years,letters):
    q=data[data.year.eq(year)]
    ax.set_facecolor("#F6F7F4")
    if land is not None: land.plot(ax=ax,color="#D5DBD7",edgecolor="#AEB7B4",linewidth=.08,zorder=0)
    zero=q.E_HW_kwh.eq(0); positive_q=q.E_HW_kwh.gt(0)&q.E_HW_kwh.notna()
    if zero.any(): ax.scatter(q.loc[zero,"centroid_x"],q.loc[zero,"centroid_y"],s=.26,c="#FFF2A8",linewidths=0,rasterized=True,zorder=1)
    artist=ax.scatter(q.loc[positive_q,"centroid_x"],q.loc[positive_q,"centroid_y"],c=q.loc[positive_q,"E_HW_kwh"],s=.30,cmap=cmap,norm=norm,linewidths=0,rasterized=True,zorder=2)
    ax.set_xlim(extent[0],extent[1]); ax.set_ylim(extent[2],extent[3]); ax.set_aspect("equal",adjustable="box"); ax.set_axis_off()
    panel_title(ax,letter,f"{year}")
cax=fig.add_axes([.920,.13,.020,.74]); cb=fig.colorbar(artist,cax=cax,extend="max")
cb.set_label("Complete-event electricity (kWh building⁻¹)")
fig.text(.46,.018,"Common logarithmic scale; cumulative values retain each event’s actual duration.",ha="center",va="bottom")
finish(fig,"Supplementary_Figure_S3_eight_event_electricity_maps",supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S4 — eight selected events at a common extent and scale.
links=pd.read_parquet(STOCK/"building_spatial_links.parquet")
links=links[links.study_region.eq("Hong Kong total")][["BUILDINGSTRUCTUREID","centroid_x","centroid_y"]]
key='dh32'; col='absolute_dh32'; den=None; label='Outage DH32 (K h)'
vmin=0; vmax=25; cmap='magma'
xmin,xmax=links.centroid_x.min(),links.centroid_x.max(); ymin,ymax=links.centroid_y.min(),links.centroid_y.max()
xpad,ypad=(xmax-xmin)*.012,(ymax-ymin)*.012
fig,axs=plt.subplots(4,2,figsize=(8.25,10.45),constrained_layout=False)
fig.subplots_adjust(left=.025,right=.900,bottom=.055,top=.975,wspace=.020,hspace=.105)
cm=plt.get_cmap(cmap).copy(); cm.set_over(cm(1.0)); norm=colors.Normalize(vmin,vmax,clip=False)
sc=None
land_path=BOUNDARIES/'housing_market_areas.geoparquet'
land=gpd.read_parquet(land_path).to_crs(2326) if land_path.exists() else None
for idx,(ax,event,year) in enumerate(zip(axs.ravel(),EVENTS,YEARS)):
    ax.set_facecolor('#F6F7F4')
    if land is not None:
        land.plot(ax=ax,color='#D5DBD7',edgecolor='#AEB7B4',linewidth=.10,zorder=0)
    columns=["BUILDINGSTRUCTUREID",col]+([den] if den else [])
    d=pd.read_parquet(EVENTS_DIR/f"{event}.parquet",columns=columns)
    d=links.merge(d,on="BUILDINGSTRUCTUREID",validate="one_to_one")
    val=d[col].to_numpy(float)

    sc=ax.scatter(d.centroid_x,d.centroid_y,c=val,s=.62,cmap=cm,norm=norm,rasterized=True,linewidths=0)
    ax.set_xlim(xmin-xpad,xmax+xpad); ax.set_ylim(ymin-ypad,ymax+ypad)
    ax.set_aspect("equal",adjustable="box"); ax.set_axis_off(); panel_title(ax,chr(97+idx),str(year))
cax=fig.add_axes([.920,.13,.020,.74]); cb=fig.colorbar(sc,cax=cax,orientation="vertical",extend="max"); cb.set_label(label)

finish(fig,'Supplementary_Figure_S4_eight_event_DH32_maps',supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S5 — full editable plotting cell.
cls = pd.read_csv(ANALYSIS / 'building_event_class_outcomes.csv')
cls['year'] = cls.event_id.map(dict(zip(EVENTS, YEARS)))
palette = {'Residential': '#1F4E79', 'Commercial': '#D55E00', 'Mixed-use': '#009E73'}
markers = {'Residential': 'o', 'Commercial': 's', 'Mixed-use': '^'}
offsets = {'Residential': -0.18, 'Commercial': 0, 'Mixed-use': 0.18}
fig, axs = plt.subplots(1, 3, figsize=(10.58, 4.55), constrained_layout=False)
fig.subplots_adjust(left=.085,right=.985,bottom=.155,top=.825,wspace=.30)
specs = [('median_shortfall_fraction_pct', 'p90_shortfall_fraction_pct', 'Cooling shortfall', 'C70 shortfall (%)'), ('median_absolute_dh32_Kh', 'p90_absolute_dh32_Kh', 'Outage overheating', 'DH32 (K h)')]
for ax, (medcol, p90col, title, ylabel) in zip(axs[:2], specs):
    for c, q in cls.groupby('main_class'):
        q = q.sort_values('year')
        x = np.arange(len(q)) + offsets[c]
        ax.vlines(x, q[medcol], q[p90col], color=palette[c], lw=1.5, alpha=0.72)
        ax.scatter(x, q[medcol], s=45, marker=markers[c], color=palette[c], edgecolor='white', linewidth=0.6, label=f'{c}: median', zorder=3)
        ax.scatter(x, q[p90col], s=40, marker=markers[c], facecolor='white', edgecolor=palette[c], linewidth=1.2, zorder=3)
    ax.set_xticks(np.arange(len(YEARS)), [str(y)[-2:] for y in YEARS], rotation=0)
    ax.set_xlabel('Event year')
    ax.set_ylabel(ylabel)
    panel_title(ax,chr(97+list(axs).index(ax)),f'{title} · median to P90')
    ax.grid(axis='y', color='#d9dbdc', lw=0.5, alpha=0.55)
for axis in axs[:2]:
    lo, hi = axis.get_ylim()
    axis.set_ylim(lo, hi + (hi - lo) * 0.35)
pass  # class and statistic encodings use the shared figure legend below
ax = axs[2]
for c, q in cls.groupby('main_class'):
    q = q.sort_values('year')
    v = q.population_degree_hours.to_numpy() / 1000000.0
    sizes = np.full(len(q), 45.0)  # fixed marker size; magnitude is encoded only by y position
    ax.plot(q.year, v, color=palette[c], lw=1.15, alpha=0.75)
    ax.scatter(q.year, v, s=sizes, marker=markers[c], color=palette[c], edgecolor='white', linewidth=0.65, label=c, zorder=3)
panel_title(ax,'c','Fixed-population heat burden')
ax.set_ylabel('million person–K h')
ax.set_xlabel('Event year')
ax.grid(axis='y', color='#d9dbdc', lw=0.5, alpha=0.55)
lo, hi = ax.get_ylim()
ax.set_ylim(lo, hi + (hi - lo) * 0.4)
class_handles=[Line2D([],[],marker=markers[c],ls="",color=palette[c],label=c) for c in ["Residential","Commercial","Mixed-use"]]
stat_handles=[Line2D([],[],marker="o",ls="",color="#555",markerfacecolor="#555",label="Median / burden value"),
              Line2D([],[],marker="o",ls="",color="#555",markerfacecolor="white",label="P90 (panels a,b)")]
fig.legend(handles=class_handles+stat_handles,frameon=False,loc="upper center",bbox_to_anchor=(.50,.985),ncol=5,
           columnspacing=.9,handletextpad=.35)
finish(fig, 'Supplementary_Figure_S5_use_class_event_profiles', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S6 — eight selected events at a common extent and scale.
links=pd.read_parquet(STOCK/"building_spatial_links.parquet")
links=links[links.study_region.eq("Hong Kong total")][["BUILDINGSTRUCTUREID","centroid_x","centroid_y"]]
key='shortfall'; col='shortfall_ref_kwh_thermal'; den='required_ref_kwh_thermal'; label='Cooling shortfall (%)'
vmin=0; vmax=15; cmap='viridis'
xmin,xmax=links.centroid_x.min(),links.centroid_x.max(); ymin,ymax=links.centroid_y.min(),links.centroid_y.max()
xpad,ypad=(xmax-xmin)*.012,(ymax-ymin)*.012
fig,axs=plt.subplots(4,2,figsize=(8.25,10.45),constrained_layout=False)
fig.subplots_adjust(left=.025,right=.900,bottom=.055,top=.975,wspace=.020,hspace=.105)
cm=plt.get_cmap(cmap).copy(); cm.set_over(cm(1.0)); norm=colors.Normalize(vmin,vmax,clip=False)
sc=None
land_path=BOUNDARIES/'housing_market_areas.geoparquet'
land=gpd.read_parquet(land_path).to_crs(2326) if land_path.exists() else None
for idx,(ax,event,year) in enumerate(zip(axs.ravel(),EVENTS,YEARS)):
    ax.set_facecolor('#F6F7F4')
    if land is not None:
        land.plot(ax=ax,color='#D5DBD7',edgecolor='#AEB7B4',linewidth=.10,zorder=0)
    columns=["BUILDINGSTRUCTUREID",col]+([den] if den else [])
    d=pd.read_parquet(EVENTS_DIR/f"{event}.parquet",columns=columns)
    d=links.merge(d,on="BUILDINGSTRUCTUREID",validate="one_to_one")
    val=np.where(d[den]>0,d[col]/d[den]*100,np.nan)

    sc=ax.scatter(d.centroid_x,d.centroid_y,c=val,s=.62,cmap=cm,norm=norm,rasterized=True,linewidths=0)
    ax.set_xlim(xmin-xpad,xmax+xpad); ax.set_ylim(ymin-ypad,ymax+ypad)
    ax.set_aspect("equal",adjustable="box"); ax.set_axis_off(); panel_title(ax,chr(97+idx),str(year))
cax=fig.add_axes([.920,.13,.020,.74]); cb=fig.colorbar(sc,cax=cax,orientation="vertical",extend="max"); cb.set_label(label)

finish(fig,'Supplementary_Figure_S6_eight_event_shortfall_maps',supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S7 — cross-event recurrence with recovery defined by status.
frames=[]
for event,year in zip(EVENTS,YEARS):
    d=pd.read_parquet(EVENTS_DIR/f"{event}.parquet",
                      columns=["BUILDINGSTRUCTUREID","absolute_dh32","recovery_status"])
    d["event_id"]=event; frames.append(d)
all_events=pd.concat(frames,ignore_index=True)
recurrence=(all_events.groupby("BUILDINGSTRUCTUREID",as_index=False)
            .agg(dh32_events=("absolute_dh32",lambda s:int(pd.to_numeric(s,errors="coerce").gt(0).sum())),
                 not_recovered_events=("recovery_status",lambda s:int(s.eq("right_censored").sum()))))
recurrence.to_csv(SD/"Supplementary_Figure_S7_recurrence_status_source.csv",index=False)
fig,axs=plt.subplots(1,2,figsize=(8.8,4.35),constrained_layout=False)
fig.subplots_adjust(left=.095,right=.925,bottom=.18,top=.86,wspace=.42)
for letter,ax,field,title in zip("ab",axs,["dh32_events","not_recovered_events"],["Events with DH32 > 0","Events right-censored at 24 h"]):
    q=recurrence[field].value_counts().reindex(range(9),fill_value=0).sort_index()
    pct=q/len(recurrence)*100; tail=pct.iloc[::-1].cumsum().iloc[::-1]
    ax.vlines(q.index,0,pct,color="#4C7899",lw=2)
    ax.scatter(q.index,pct,s=55,color="#1F4E79",edgecolor="white",linewidth=.6,label="Exact frequency")
    ax2=ax.twinx(); ax2.plot(q.index,tail,marker="D",ms=4.5,color="#D55E00",lw=1.35,label="At least this many events")
    ax2.set_ylabel("Cumulative buildings (%)",color="#D55E00")
    ax.set_xticks(range(9)); ax.set_xlabel("Number of eight events"); ax.set_ylabel("Buildings (%)",color="#1F4E79")
    ax.set_ylim(0,100); ax2.set_ylim(0,100)
    panel_title(ax,letter,title)
    handles=[Line2D([],[],marker="o",ls="",color="#1F4E79",label="Exact frequency"),Line2D([],[],marker="D",ls="-",color="#D55E00",label="Cumulative tail")]
    ax.legend(handles=handles,frameon=False,loc="center",bbox_to_anchor=(.52,.43))
finish(fig,'Supplementary_Figure_S7_cross_event_recurrence',supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S8 — full editable plotting cell.
city = pd.read_csv(ANALYSIS / 'city_event_energy_and_peak_definitions.csv')
city['year'] = city.event_id.map(dict(zip(EVENTS, YEARS)))
y = np.arange(len(city))
fig, axs = plt.subplots(1, 3, figsize=(10.58, 4.70), constrained_layout=False)
fig.subplots_adjust(left=.075,right=.985,bottom=.165,top=.865,wspace=.34)
ax = axs[0]
pos = city.positive_timestep_delta_kwh / 1000000.0
neg = city.negative_timestep_delta_kwh / 1000000.0
net = city.net_delta_gwh
ax.hlines(y, neg, pos, color='#A9B6BF', lw=2)
ax.scatter(pos, y, s=50, color='#D55E00', label='+ time steps')
ax.scatter(neg, y, s=50, color='#4C78A8', label='− time steps')
ax.scatter(net, y, s=55, marker='D', facecolor='white', edgecolor='#1F4E79', linewidth=1.4, label='Net')
ax.axvline(0, color='#666', lw=0.7)
ax.set_yticks(y, city.year.astype(str))
ax.set(xlabel='Event electricity (GWh)', ylabel='Event year'); panel_title(ax,'a','Signed time-step contributions and net')
lo, hi = ax.get_xlim()
ax.set_xlim(lo, hi + (hi - lo) * 0.55)
ax.legend(frameon=False, loc='upper right')
ax = axs[1]
same = city.same_time_difference_at_hw_peak_gw
sep = city.difference_of_separate_maxima_gw
ax.hlines(y, np.minimum(same, sep), np.maximum(same, sep), color='#B8C2CA', lw=2)
ax.scatter(same, y, s=52, marker='D', facecolor='white', edgecolor='#C94C62', linewidth=1.4, label='Same time')
ax.scatter(sep, y, s=48, marker='s', color='#C94C62', edgecolor='white', linewidth=0.6, label='Separate maxima')
ax.axvline(0, color='#666', lw=0.7)
ax.set_yticks(y, city.year.astype(str))
ax.set(xlabel='Peak-demand difference (GW)'); panel_title(ax,'b','Peak definitions and time displacement')
lo, hi = ax.get_xlim()
ax.set_xlim(lo, hi + (hi - lo) * 0.55)
ax.legend(frameon=False, loc='upper right')
ax = axs[2]
hw = city.hw_absolute_peak_gw
cf_same = city.simultaneous_CF_peak_kw / 1000000.0
cf_max = hw - city.difference_of_separate_maxima_gw
for yi, a, b, c in zip(y, cf_same, cf_max, hw):
    ax.hlines(yi, min(a, b, c), max(a, b, c), color='#B8C2CA', lw=1.8)
ax.scatter(cf_same, y, s=44, marker='o', facecolor='white', edgecolor='#6B7280', linewidth=1.2, label='Reference at HW peak')
ax.scatter(cf_max, y, s=48, marker='s', facecolor='white', edgecolor='#0072B2', linewidth=1.3, label='Reference maximum')
ax.scatter(hw, y, s=52, marker='D', color='#D55E00', edgecolor='white', linewidth=0.6, label='Heatwave maximum')
ax.set_yticks(y, city.year.astype(str))
ax.set(xlabel='Modelled demand (GW)'); panel_title(ax,'c','Three absolute demand landmarks')
lo, hi = ax.get_xlim()
ax.set_xlim(lo, hi + (hi - lo) * 2.20)
ax.legend(frameon=False, loc='lower right', bbox_to_anchor=(.985,.025))
finish(fig, 'Supplementary_Figure_S8_complete_event_energy_metrics', supplementary=True)


In [ ]:
from pathlib import Path
from zoneinfo import ZoneInfo
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S9 — full editable plotting cell.
p = CITY_TRAJECTORIES / 'HKO_LI_0052_city_15min.parquet'
if not p.exists():
    p = CITY_TRAJECTORIES / 'HKO_LI_0052_city_15min.parquet'
tr = pd.read_parquet(p)
fig, ax = plt.subplots(figsize=(6.7, 3.55))
HKT=ZoneInfo("Asia/Hong_Kong")
t = pd.to_datetime(tr.timestamp_hkt,utc=True).dt.tz_convert(HKT)
known_input=pd.Timestamp("2022-07-19 00:15",tz=HKT)
assert mdates.num2date(mdates.date2num(known_input.to_pydatetime()),tz=HKT).strftime("%d %b %H:%M")=="19 Jul 00:15"
h = [c for c in tr.columns if c.lower() in {'power_kw_hw', 'p_hw_kw', 'hw_power_kw'}][0]
c = [c for c in tr.columns if c.lower() in {'power_kw_cf', 'p_cf_kw', 'cf_power_kw'}][0]
hw = tr[h].to_numpy() / 1000000.0
cf = tr[c].to_numpy() / 1000000.0
delta = hw - cf
ax.fill_between(t, hw, cf, where=delta >= 0, color='#D55E00', alpha=0.16, interpolate=True, label='Positive difference')
ax.fill_between(t, hw, cf, where=delta < 0, color='#4C78A8', alpha=0.13, interpolate=True, label='Negative difference')
ax.plot(t, hw, label='Heatwave', color='#D55E00', lw=1.45)
ax.plot(t, cf, label='Reference', color='#0072B2', lw=1.35)
ih = int(np.nanargmax(hw))
ic = int(np.nanargmax(cf))
ax.scatter([t.iloc[ih], t.iloc[ic]], [hw[ih], cf[ic]], s=66, marker='D', facecolor='white', edgecolor=['#D55E00', '#0072B2'], linewidth=1.4, zorder=5)
ax.vlines([t.iloc[ih], t.iloc[ic]], [min(hw.min(), cf.min())] * 2, [hw[ih], cf[ic]], color=['#D55E00', '#0072B2'], ls='--', lw=0.8, alpha=0.65)
ax.set(ylabel='Modelled demand (GW)', title='2009 demand trajectory, signed differences and non-coincident maxima')
ax.xaxis.set_major_locator(mdates.DayLocator(tz=HKT))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b',tz=HKT))
lo, hi = ax.get_ylim()
ax.set_ylim(lo, hi + (hi - lo) * 0.35)
ax.legend(frameon=False, ncol=3, loc='upper left')
ax.tick_params(axis='x', rotation=0)
finish(fig, 'Supplementary_Figure_S9_2009_peak_definition', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S10 — full editable plotting cell.
d = pd.read_csv(ANALYSIS / 'mechanism_analysis' / 'controlled_pair_effects.csv')
fig, ax = plt.subplots(figsize=(6.7, 4.1))
factor_label={'roof_top_exposure':'Roof-top exposure','thermal_mass':'Thermal mass','nominal_cooling_capacity':'Cooling capacity','AC_operating_time':'AC schedule phase shift'}
for f, q in d.groupby('factor'):
    ax.scatter(q.effect_shortfall_ref_kwh_thermal, q.effect_absolute_occupied_degree_hours_above_32c, s=15, alpha=0.55, label=factor_label[f], color=FACTOR_COLORS.get(f, '#777'))
ax.axhline(0, color='#777', lw=0.6)
ax.axvline(0, color='#777', lw=0.6)
ax.set(xlabel='Δ cooling shortfall (kWhₜₕ)', ylabel='Δ DH32 (K h)', title='All 105 paired mechanism effects')
xlo, xhi = ax.get_xlim()
ylo, yhi = ax.get_ylim()
ax.set_xlim(xlo, xhi + (xhi - xlo) * 0.32)
ax.set_ylim(ylo, yhi + (yhi - ylo) * 0.35)
ax.legend(frameon=False, loc='upper right')
finish(fig, 'Supplementary_Figure_S10_all_mechanism_effects', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S11 — full editable plotting cell.
rank = pd.read_csv(ANALYSIS / 'loeo8_lower_rank_summary.csv')
methods = ['BERI', 'stage_C', 'stage_O', 'stage_R', 'detailed_use_floor']
chosen = rank[rank.method.isin(methods)].copy()
outcomes = chosen.outcome.drop_duplicates().tolist()
outcome_label={'electricity_increment':'Electricity difference','cooling_shortfall_kwh':'Cooling shortfall','absolute_dh32':'Outage DH32','extra_dh26':'Added DH26','population_degree_hours':'Population heat','recovery_hours':'Recovery time'}
method_label={'BERI':'BERI','stage_C':'Cooling-service score','stage_O':'Outage-tolerance score','stage_R':'Recovery score','detailed_use_floor':'Detailed use + floor'}
fig, ax = plt.subplots(figsize=(7.2, 5.45), constrained_layout=False)
fig.subplots_adjust(left=.24,right=.98,bottom=.13,top=.88)
markers = ['o', 's', '^', 'D', 'P']
palette = ['#2C7BB6', '#0F766E', '#CC79A7', '#D55E00', '#E69F00']
y = np.arange(len(outcomes))
offsets = np.linspace(-0.24, 0.24, len(methods))
for offset, marker, color, method in zip(offsets, markers, palette, methods):
    q = chosen[chosen.method.eq(method)].set_index('outcome').reindex(outcomes)
    ax.hlines(y + offset, q.min_spearman, q.max_spearman, color=color, lw=1.4, alpha=0.62)
    ax.scatter(q.median_spearman, y + offset, marker=marker, s=58, facecolor='white', edgecolor=color, linewidth=1.5, label=method_label[method], zorder=3)
ax.axvline(0, color='#777777', lw=0.8)
ax.set_yticks(y, [outcome_label.get(v,v.replace('_',' ')) for v in outcomes])
ax.set_ylim(-0.50, len(outcomes) + 1.20)
ax.set_xlabel('Spearman ρ: median and min–max across eight LOEO folds')
ax.grid(axis='x', color='#d9dbdc', linewidth=0.5, alpha=0.6)
lo, hi = ax.get_xlim()
ax.set_xlim(min(lo, -0.05), 1.50)
ax.legend(frameon=False, ncol=2, loc='upper right', bbox_to_anchor=(.995,.985), columnspacing=.75, handletextpad=.35, labelspacing=.35)
ax.set_title('LOEO8 lower-endpoint rank agreement')
finish(fig, 'Supplementary_Figure_S11_full_screening', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S12 — full editable plotting cell.
rating = pd.read_csv(VALIDATION / 'full_event_reference_rating.csv')
fig, ax = plt.subplots(figsize=(6.7, 3.85), constrained_layout=False)
fig.subplots_adjust(left=.14,right=.98,bottom=.22,top=.86)
cols = ['score_N_lower', 'score_C_lower', 'score_O_lower', 'score_R_lower', 'BERI_lower']
labels = ['N', 'C', 'O', 'R', 'BERI']
colors = ['#56B4E9', '#0072B2', '#CC79A7', '#D55E00', '#222222']
values = [rating[c].to_numpy() * (100 if c != 'BERI_lower' else 1) for c in cols]
pos = np.arange(1, 6)
vp = ax.violinplot(values, positions=pos, vert=False, widths=0.72, showextrema=False)
for body, color in zip(vp['bodies'], colors):
    body.set_facecolor(color)
    body.set_edgecolor(color)
    body.set_alpha(0.22)
rng = np.random.default_rng(4)
for yi, (v, color, c) in enumerate(zip(values, colors, cols), start=1):
    q25, med, q75 = np.nanpercentile(v, [25, 50, 75])
    ax.hlines(yi, q25, q75, color=color, lw=5, alpha=0.75, zorder=3)
    ax.scatter(med, yi, s=55, marker='D', facecolor='white', edgecolor=color, linewidth=1.5, zorder=4)
    zero = (rating[c] == 0).mean() * 100
    ax.scatter(0, yi, s=28 + zero * 2.2, facecolor='none', edgecolor=color, linewidth=1.2, zorder=5)
    ax.text(101.5, yi, f'zero {zero:.1f}%', va='center', fontsize=11, color='black')
ax.set_yticks(pos, labels)
ax.set_xlim(-3, 118)
ax.set(xlabel='Lower-endpoint score (0–100)', title='Score distributions, quartile spans and exact-zero mass')
ax.grid(axis='x', color='#d9dbdc', lw=0.5, alpha=0.55)
finish(fig, 'Supplementary_Figure_S12_BERI_zero_ties', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S13 — full editable plotting cell.
sens = pd.read_csv(VALIDATION / 'sensitivity_temperature_thresholds.csv')
s = sens.groupby(['event_id', 'threshold_c'], as_index=False).degree_hours.median()
s['year'] = event_year(s.event_id)
fig, ax = plt.subplots(figsize=(7.2, 4.15), constrained_layout=False)
fig.subplots_adjust(left=.14,right=.98,bottom=.18,top=.86)
y = np.arange(len(YEARS))
markers = {30: 'o', 32: 'D', 34: 's'}
colors = {30: '#D55E00', 32: '#CC79A7', 34: '#0072B2'}
for yi, year in enumerate(YEARS):
    q = s[s.year.eq(year)].set_index('threshold_c').reindex([30, 32, 34])
    ax.hlines(yi, q.degree_hours.min(), q.degree_hours.max(), color='#B8C2CA', lw=1.5, zorder=1)
    for th in [30, 32, 34]:
        ax.scatter(q.loc[th, 'degree_hours'], yi, s=52, marker=markers[th], facecolor='white', edgecolor=colors[th], linewidth=1.5, zorder=3)
ax.set_yticks(y, [str(v) for v in YEARS])
ax.set(xlabel='Design-level median occupied degree-hours (K h)', ylabel='Event year', title='HW_O24_SYNC threshold sensitivity across complete events')
lo, hi = ax.get_xlim()
ax.set_xlim(lo, hi + (hi - lo) * 0.35)
ax.legend(handles=[Line2D([], [], marker=markers[t], ls='', markerfacecolor='white', markeredgecolor=colors[t], label=f'>{t} °C') for t in [30, 32, 34]], frameon=False, ncol=1, loc='lower right')
ax.grid(axis='x', color='#d9dbdc', lw=0.5, alpha=0.55)
finish(fig, 'Supplementary_Figure_S13_temperature_thresholds', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S14 — full editable plotting cell.
rec = pd.read_csv(ANALYSIS / 'design_level_recovery_comparator.csv')
rec['year'] = rec.event_id.map(dict(zip(EVENTS, YEARS)))
fig, axs = plt.subplots(1, 2, figsize=(6.7, 3.7), constrained_layout=True)
for ax, c in zip(axs, ['Residential', 'Commercial']):
    q = rec[rec.sector.eq(c)].sort_values('year')
    y = np.arange(len(q))
    o = q.outage_right_censored / q.design_groups * 100
    r = q.rated_right_censored / q.design_groups * 100
    ax.hlines(y, np.minimum(o, r), np.maximum(o, r), color='#A9B6BF', lw=2)
    ax.scatter(o, y, s=55, color='#B2182B', label='O24')
    ax.scatter(r, y, s=55, marker='D', facecolor='white', edgecolor='#2166AC', linewidth=1.4, label='RATED')
    ax.set_yticks(y, q.year.astype(str))
    ax.set_title(c)
    ax.set_xlabel('Design groups right-censored (%)')
    ax.set_ylabel('Event year')
    ax.grid(axis='x', color='#d9dbdc', lw=0.5, alpha=0.55)
    ax.set_xlim(0, 135)
    ax.legend(frameon=False, loc='upper right')
finish(fig, 'Supplementary_Figure_S14_recovery_outage_rated_comparator', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S15 — full editable plotting cell.
w = pd.read_csv(VALIDATION / 'sensitivity_beri_weights.csv')
labels=[row.weight_scenario.replace('_',' ') for row in w.itertuples(index=False)]
y=np.arange(len(w))
values=w.median_BERI_lower.to_numpy(float)
fig, ax = plt.subplots(figsize=(6.7, 3.7), constrained_layout=False)
fig.subplots_adjust(left=.27,right=.96,bottom=.18,top=.86)
colors = plt.cm.cividis(np.linspace(0.12, 0.88, len(w)))
ax.hlines(y,0,values,color=colors,lw=2.0,alpha=.70)
ax.scatter(values,y,s=72,marker='D',facecolor='white',edgecolor=colors,linewidth=1.6,zorder=4)
for yi,value in zip(y,values):
    ax.text(value+1.8,yi,f'{value:.3f}',va='center',ha='left')
ax.set_yticks(y,labels)
ax.set_xlabel('Stock-median lower-endpoint BERI score (0–100)')
ax.set_title('Lower-endpoint score under prespecified weight scenarios')
ax.set_xlim(0,max(60,values.max()+12)); ax.invert_yaxis()
ax.grid(axis='x',color='#d9dbdc',lw=.5,alpha=.55)
finish(fig, 'Supplementary_Figure_S15_BERI_weight_sensitivity', supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S16 — LOEO8 screening-fraction sensitivity by outcome.
cap=pd.read_csv(VALIDATION/"event_holdout_screening_capture.csv")
cap=cap[cap.fold_id.str.startswith("LOEO_") & cap.score_endpoint_scenario.eq("lower") & cap.fraction.isin([.05,.10,.20])].copy()
methods=[m for m in ["BERI","stage_C","stage_O","stage_R","detailed_use_floor"] if m in set(cap.method)]
outcomes=["cooling_shortfall_kwh","absolute_dh32","extra_dh26","population_degree_hours","recovery_hours"]
labels={"cooling_shortfall_kwh":"Cooling shortfall","absolute_dh32":"Outage DH32","extra_dh26":"Outage-added DH26","population_degree_hours":"Population heat burden","recovery_hours":"Recovery time"}
palette=dict(zip(methods,["#2C7BB6","#0F766E","#CC79A7","#D55E00","#E69F00"]))
summary=(cap[cap.method.isin(methods)&cap.outcome.isin(outcomes)]
         .groupby(["outcome","fraction","method"]).expected_burden_capture.agg(["median","min","max"]).reset_index())
summary.to_csv(SD/"Supplementary_Figure_S16_LOEO8_lower_fraction_sensitivity.csv",index=False)
fig,axs=plt.subplots(2,3,figsize=(10.58,7.15),constrained_layout=False)
fig.subplots_adjust(left=.075,right=.985,bottom=.10,top=.94,wspace=.30,hspace=.38)
for idx,(ax,outcome) in enumerate(zip(axs.ravel()[:5],outcomes)):
    for method in methods:
        z=summary[(summary.outcome.eq(outcome))&(summary.method.eq(method))].sort_values("fraction")
        x=z.fraction.to_numpy()*100
        ax.fill_between(x,z["min"].to_numpy()*100,z["max"].to_numpy()*100,color=palette[method],alpha=.10)
        ax.plot(x,z["median"].to_numpy()*100,marker="o",lw=1.35,color=palette[method],label=method.replace("_"," "))
    ax.plot([5,20],[5,20],ls="--",color="#777",lw=.9,label="Random expectation")
    ax.set_xticks([5,10,20]); ax.set_xlabel("Buildings selected (%)"); ax.set_ylabel("Burden captured (%)")
    ax.grid(color="#D9DDE0",lw=.5,alpha=.55); panel_title(ax,chr(97+idx),labels[outcome])
    lo,hi=ax.get_ylim(); ax.set_ylim(lo,hi+(hi-lo)*.18)
legend_ax=axs.ravel()[5]; legend_ax.axis("off")
handles=[Line2D([],[],marker="o",color=palette[m],label=m.replace("_"," ")) for m in methods]
handles.append(Line2D([],[],ls="--",color="#777",label="Random expectation"))
legend_ax.legend(handles=handles,frameon=False,loc="center",ncol=1,title="LOEO8 · lower endpoint")
finish(fig,"Supplementary_Figure_S16_screening_fraction_sensitivity",supplementary=True)


In [ ]:
from pathlib import Path
import json, math, re, textwrap
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import dates as mdates
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from shapely import from_wkb

REPO = Path.cwd()
if not (REPO / "result" / "figure_source_data").exists():
    candidate = REPO.parent
    if (candidate / "result" / "figure_source_data").exists():
        REPO = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or plot directory.")
SD = REPO / "result" / "figure_source_data"
ANALYSIS = REPO / "result" / "analysis"
STOCK = REPO / "intermediate" / "prepared_stock"
EVENTS_DIR = REPO / "result" / "building_event_outcomes"
CITY_TRAJECTORIES = REPO / "intermediate" / "city_trajectories"
VALIDATION = REPO / "result" / "validation"
BOUNDARIES = REPO / "intermediate" / "spatial" / "boundaries"
ROADS = REPO / "intermediate" / "spatial" / "roads"
HAZ = VALIDATION
FIGS = REPO / "output" / "figures"
SUPP = FIGS
FIGS.mkdir(parents=True, exist_ok=True)

FONT = "Times New Roman"
plt.rcParams.update({
    "font.family": "serif", "font.serif": [FONT], "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.fontsize": 11, "legend.title_fontsize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

TYPE_COLORS = {
    "Public rental": "#174A7E", "Subsidised sale": "#2F75B5",
    "Private high-rise": "#5B9BD5", "Old low-rise": "#75B7D3",
    "Village house": "#A6CEE3", "Office": "#A50F15", "Retail": "#DE2D26",
    "Restaurant": "#FB6A4A", "Accommodation": "#C51B7D",
    "Education": "#88419D", "Healthcare": "#E6550D", "Data centre": "#F16913",
    "Other commercial": "#FDAE6B", "Mixed-use": "#4D4D4D",
}
CLASS_COLORS = {"Residential": "#2F75B5", "Mixed-use": "#E69F00", "Commercial": "#D95F4A"}
EVENT_IDS = ["HKO_LI_0049","HKO_LI_0052","HKO_LI_0054","HKO_LI_0058",
             "HKO_LI_0066","HKO_LI_0068","HKO_LI_0085","HKO_LI_0090"]
EVENT_YEARS = [2005,2009,2012,2014,2017,2018,2022,2023]
YEAR_OF = dict(zip(EVENT_IDS, EVENT_YEARS))
EVENTS = EVENT_IDS
YEARS = EVENT_YEARS
FACTOR_COLORS = {"roof_top_exposure":"#D55E00", "thermal_mass":"#009E73",
                 "nominal_cooling_capacity":"#0072B2", "AC_operating_time":"#9C4F96"}

def event_year(series):
    return series.map(YEAR_OF)

def panel_title(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left", pad=6, fontsize=12, fontweight="bold", color="black")

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    return list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]

def polygon_layer(ax, geoms, values, *, cmap=None, norm=None, categories=None,
                  edge="#7E8788", linewidth=.045, rasterized=True):
    patches, numerical, faces = [], [], []
    for geom, value in zip(geoms, values):
        for part in polygon_parts(geom):
            xy = np.asarray(part.exterior.coords)
            if len(xy) < 3:
                continue
            patches.append(MplPolygon(xy, closed=True))
            if categories is None:
                numerical.append(value)
            else:
                faces.append(categories.get(value, "#D9D9D9"))
    layer = PatchCollection(patches, linewidths=linewidth, edgecolors=edge, rasterized=rasterized)
    if categories is None:
        layer.set_cmap(cmap); layer.set_norm(norm); layer.set_array(np.asarray(numerical, dtype=float))
    else:
        layer.set_facecolor(faces)
    ax.add_collection(layer)
    ax.autoscale_view(); ax.set_aspect("equal", adjustable="box"); ax.set_axis_off()
    return layer

def finish(fig, stem, supplementary=False):
    destination = SUPP if supplementary else FIGS
    for text in fig.findobj(match=lambda obj: hasattr(obj, "set_fontfamily")):
        try:
            text.set_fontfamily(FONT); text.set_color("black")
        except Exception:
            pass
    fig.savefig(destination / f"{stem}.png", dpi=450, facecolor="white")
    plt.show()

# SUPPLEMENTARY FIGURE S17 — full editable plotting cell.
dn = pd.read_csv(VALIDATION / 'day_night_humid_recovery.csv')
metrics = [('temperature_mean_c', 'Mean temperature (°C)'), ('relative_humidity_mean_pct', 'Mean relative humidity (%)'), ('same_zone_heat_index_mean_c', 'Same-zone heat index (°C)'), ('absolute_dh32_k_h', 'Absolute DH32 (K h)')]
fig, axs = plt.subplots(2, 2, figsize=(6.7, 5.45), constrained_layout=True)
rng = np.random.default_rng(8)
periods = ['day', 'night']
pcolors = {'day': '#D55E00', 'night': '#1F4E79'}
for panel_index, (ax, (metric, label)) in enumerate(zip(axs.ravel(), metrics)):
    arrays = [dn.loc[dn.period_id.eq(p), metric].dropna().to_numpy() for p in periods]
    vp = ax.violinplot(arrays, positions=[1, 2], widths=0.72, showextrema=False)
    for body, p in zip(vp['bodies'], periods):
        body.set_facecolor(pcolors[p])
        body.set_edgecolor(pcolors[p])
        body.set_alpha(0.22)
    tick_labels = []
    for xi, p, a in zip([1, 2], periods, arrays):
        q25, med, q75 = np.nanpercentile(a, [25, 50, 75])
        ax.vlines(xi, q25, q75, color=pcolors[p], lw=5, alpha=0.75)
        ax.scatter(xi, med, s=55, marker='D', facecolor='white', edgecolor=pcolors[p], linewidth=1.4, zorder=4)
        ev = dn[dn.period_id.eq(p)].groupby('event_id')[metric].median().dropna()
        ax.scatter(np.full(len(ev), xi) + rng.uniform(-0.11, 0.11, len(ev)), ev, s=24, color=pcolors[p], edgecolor='white', linewidth=0.4, zorder=3)
        tick_labels.append(f'{p.title()}\nn={len(a):,}; events={len(ev)}')
    ax.set_xticks([1, 2], tick_labels)
    ax.set_ylabel(label)
    panel_title(ax,chr(97+panel_index),label)
    ax.grid(axis='y', color='#d9dbdc', lw=0.5, alpha=0.55)
finish(fig, 'Supplementary_Figure_S17_day_night_humidity_recovery', supplementary=True)
